# Visualización I · Sesión 29
## Streamlit: diseño narrativo y control de interacción
**Universidad Externado de Colombia · Programa de Ciencia de Datos**  
**Docente:** Adriana Pineda  
**Entorno:** Google Colab + aplicación Streamlit

### Propósito
Construir **paso a paso** una aplicación web que permita explorar datos mediante controles interactivos y gráficos Plotly, y comunicar hallazgos a una audiencia no técnica.

**Producto de la sesión:** `app_streamlit.py` (aplicación) y `ventas_streamlit.csv` (datos).  
**Ruta didáctica:** comprender → preparar datos → construir interfaz → integrar gráficos → probar → reflexionar.

> **Importante:** Las celdas de Colab generan y ejecutan un archivo `.py`. La aplicación se abre en otra pestaña mediante el proxy de Colab; **no** se representa dentro de la salida normal del notebook. Ejecuta las celdas **en orden**.

## 0. ¿Qué aprenderemos?
Al finalizar podrás:
1. Diferenciar un gráfico interactivo Plotly de una **aplicación** Streamlit.
2. Usar `selectbox`, `slider`, `button` y controles de entrada.
3. Vincular los controles con indicadores y visualizaciones.
4. Construir una narrativa: **qué ocurrió → dónde ocurrió → qué decisión sugiere**.
5. Documentar y reproducir la aplicación.

**Caso didáctico:** ventas simuladas de una cadena comercial con tiendas en cuatro ciudades colombianas, de enero a diciembre de 2025. Son **datos sintéticos**: no corresponden a una empresa real.

**Pregunta para discutir:** ¿Qué puede hacer una audiencia con una aplicación que no puede hacer con una imagen de un gráfico?

## 1. Preparación del entorno
**Paso 1.** Ejecuta la siguiente celda. Instala Streamlit y las bibliotecas que necesita la aplicación. La instalación puede tardar unos minutos.

> Si Colab pide reiniciar el entorno después de instalar, reinícialo y vuelve a ejecutar desde aquí.

In [13]:
%pip -q install streamlit plotly pandas numpy

## 2. Construcción de la base de datos
Para que el material funcione **sin archivos externos**, generaremos una base sintética reproducible.

**Paso 2.** Ejecuta la celda y revisa las primeras filas. `np.random.default_rng(29)` permite obtener siempre los mismos datos.

**Variables:** `fecha`, `ciudad`, `categoria`, `unidades`, `precio_unitario`, `ventas`.

In [14]:
import numpy as np
import pandas as pd
from IPython.display import display

rng = np.random.default_rng(29)
n = 720
fechas = pd.to_datetime(rng.choice(pd.date_range("2025-01-01", "2025-12-31"), n))
ciudades = rng.choice(["Bogotá", "Medellín", "Cali", "Barranquilla"], n, p=[.4, .25, .2, .15])
categorias = rng.choice(["Tecnología", "Hogar", "Papelería"], n, p=[.35, .4, .25])
unidades = rng.integers(1, 9, n)
precios = np.array([{"Tecnología": 185000, "Hogar": 75000, "Papelería": 18000}[c] for c in categorias])
precios = np.round(precios * rng.uniform(.75, 1.30, n), -2).astype(int)
datos = pd.DataFrame({"fecha": fechas, "ciudad": ciudades, "categoria": categorias,
                      "unidades": unidades, "precio_unitario": precios})
datos["ventas"] = datos["unidades"] * datos["precio_unitario"]
datos = datos.sort_values("fecha").reset_index(drop=True)
datos.to_csv("ventas_streamlit.csv", index=False, encoding="utf-8-sig")
display(datos.head())
print("Registros:", len(datos), "| Ventas totales:", f"${datos['ventas'].sum():,.0f}")

,fecha,ciudad,categoria,unidades,precio_unitario,ventas
0,2025-01-01,Cali,Hogar,7,93900,657300
1,2025-01-01,Medellín,Papelería,1,20200,20200
2,2025-01-01,Medellín,Tecnología,6,180200,1081200
3,2025-01-01,Barranquilla,Hogar,6,79800,478800
4,2025-01-02,Bogotá,Papelería,2,23300,46600


Registros: 720 | Ventas totales: $333,424,900


### 2.1. Comprobación de calidad
Antes de construir gráficos, validamos tipos, nulos, duplicados y coherencia de la variable de ventas.

**Pregunta:** ¿Qué ocurriría si `fecha` fuera texto y quisiéramos agrupar por mes?

In [15]:
print(datos.dtypes)
print("\nNulos por columna:\n", datos.isna().sum())
print("\nDuplicados exactos:", datos.duplicated().sum())
assert datos["ventas"].ge(0).all()
assert (datos["ventas"] == datos["unidades"] * datos["precio_unitario"]).all()
print("\nValidaciones superadas.")

fecha              datetime64[ns]
ciudad                     object
categoria                  object
unidades                    int64
precio_unitario             int64
ventas                      int64
dtype: object

Nulos por columna:
 fecha              0
ciudad             0
categoria          0
unidades           0
precio_unitario    0
ventas             0
dtype: int64

Duplicados exactos: 0

Validaciones superadas.


## 3. ¿Cómo se construye una aplicación Streamlit?
En Plotly creamos una figura; en Streamlit construimos una **interfaz** que muestra figuras, indicadores y controles.

Una aplicación Streamlit es un archivo Python (`.py`). Cada interacción suele provocar una nueva ejecución del script de arriba hacia abajo. Por ello, **los filtros deben definirse antes de los cálculos y gráficos**.

En Colab usaremos `%%writefile` para crear el archivo. En las siguientes secciones **añadiremos código al mismo archivo** con `%%writefile -a`, sin sustituir lo anterior.

**Regla:** si ejecutas una sección dos veces, duplicarás ese fragmento en el archivo. Para reconstruirlo, ejecuta nuevamente desde la sección 4, en orden.

## 4. Primera aplicación: título y datos
**Paso 3.** Esta celda **prepara el primer bloque** de `app_streamlit.py`. El archivo completo se escribirá una sola vez en la sección 7.1. Revisa las funciones de `st.title`, `st.caption` y `st.dataframe`.

**Observación:** `st.set_page_config` debe ejecutarse antes de mostrar componentes de la interfaz.

In [16]:
# BLOQUE DIDÁCTICO: bloque_inicio
# Se guarda como texto Python; todavía NO se modifica app_streamlit.py.
bloque_inicio = 'import streamlit as st\nimport pandas as pd\nimport plotly.express as px\n\nst.set_page_config(page_title="Ventas | Visualización I", page_icon="📊", layout="wide")\nst.title("Ventas 2025 | Exploración interactiva")\nst.caption("Caso didáctico con datos sintéticos · Visualización I")\n\n# Guardamos la lectura en caché para evitar cargar el CSV en cada interacción.\n@st.cache_data\ndef cargar_datos():\n    df = pd.read_csv("ventas_streamlit.csv", parse_dates=["fecha"])\n    df["mes"] = df["fecha"].dt.to_period("M").astype(str)\n    return df\n\ndf = cargar_datos()\nst.write("**Contexto:** registros de ventas simuladas de cuatro ciudades colombianas.")\nwith st.expander("Ver datos originales"):\n    st.dataframe(df, use_container_width=True)\n'
print("Bloque preparado:", "bloque_inicio")

Bloque preparado: bloque_inicio


## 5. Controles de usuario
**Paso 4.** Añadimos una barra lateral con `selectbox`, `multiselect`, `slider` y un `text_input`. Todos los gráficos usarán la misma tabla filtrada.

**Importante:** el botón de restablecimiento se implementa con `st.button` y `st.rerun`, usando `session_state` para conservar el estado de los filtros.

In [17]:
# BLOQUE DIDÁCTICO: bloque_filtros
# Se guarda como texto Python; todavía NO se modifica app_streamlit.py.
bloque_filtros = '# --- Controles de la barra lateral ---\n# session_state conserva las selecciones entre las actualizaciones de Streamlit.\nst.sidebar.header("Explorar los datos")\nciudades = sorted(df["ciudad"].unique().tolist())\ncategorias = sorted(df["categoria"].unique().tolist())\nmeses = sorted(df["fecha"].dt.month.unique().tolist())\n\nif "ciudad_sel" not in st.session_state:\n    st.session_state.ciudad_sel = "Todas"\nif "categorias_sel" not in st.session_state:\n    st.session_state.categorias_sel = categorias.copy()\nif "mes_max" not in st.session_state:\n    st.session_state.mes_max = max(meses)\nif "busqueda" not in st.session_state:\n    st.session_state.busqueda = ""\n\ndef restablecer():\n    st.session_state.ciudad_sel = "Todas"\n    st.session_state.categorias_sel = categorias.copy()\n    st.session_state.mes_max = max(meses)\n    st.session_state.busqueda = ""\n\nst.sidebar.selectbox("Ciudad", ["Todas"] + ciudades, key="ciudad_sel")\nst.sidebar.multiselect("Categorías", categorias, key="categorias_sel")\nst.sidebar.slider("Mostrar hasta el mes", min_value=min(meses),\n                  max_value=max(meses), key="mes_max")\nst.sidebar.text_input("Buscar texto en la categoría", key="busqueda")\nst.sidebar.button("Restablecer filtros", on_click=restablecer)\n\n# Aplicamos los cuatro filtros sobre una misma copia de los datos.\n# Ningún indicador o gráfico debe usar df sin filtrar.\nfiltrado = df.copy()\nif st.session_state.ciudad_sel != "Todas":\n    filtrado = filtrado[filtrado["ciudad"] == st.session_state.ciudad_sel]\nfiltrado = filtrado[filtrado["categoria"].isin(st.session_state.categorias_sel)]\nfiltrado = filtrado[filtrado["fecha"].dt.month <= st.session_state.mes_max]\nif st.session_state.busqueda.strip():\n    filtrado = filtrado[filtrado["categoria"].str.contains(\n        st.session_state.busqueda.strip(), case=False, regex=False, na=False\n    )]\n\nst.info(f"Registros seleccionados: {len(filtrado)} de {len(df)}")\nif filtrado.empty:\n    st.warning("No hay registros con estos filtros. Modifica la selección o restablece.")\n    st.stop()\n'
print("Bloque preparado:", "bloque_filtros")

Bloque preparado: bloque_filtros


## 6. Indicadores y gráficos que responden a los filtros
**Paso 5.** Integramos tres indicadores y dos visualizaciones Plotly.

- **Indicadores:** ventas totales, unidades y valor medio por registro.
- **Gráfico 1:** evolución mensual de ventas.
- **Gráfico 2:** ventas por categoría.

**Para discutir:** el promedio por registro **no** es el precio medio de un producto. ¿Por qué?

In [18]:
# BLOQUE DIDÁCTICO: bloque_visualizaciones
# Se guarda como texto Python; todavía NO se modifica app_streamlit.py.
bloque_visualizaciones = '# --- Indicadores: se calculan UNA sola vez, después de aplicar TODOS los filtros ---\nst.subheader("1 · Panorama general")\nc1, c2, c3 = st.columns(3)\nc1.metric("Ventas totales", f"$ {filtrado[\'ventas\'].sum():,.0f}")\nc2.metric("Unidades vendidas", f"{filtrado[\'unidades\'].sum():,}")\nc3.metric("Venta promedio por registro", f"$ {filtrado[\'ventas\'].mean():,.0f}")\n\n# --- Gráficos: emplean el MISMO DataFrame filtrado que los indicadores ---\nst.subheader("2 · ¿Cómo evolucionaron las ventas?")\nserie = (filtrado.groupby("mes", as_index=False)["ventas"]\n         .sum().sort_values("mes"))\nfig_serie = px.line(serie, x="mes", y="ventas", markers=True,\n                    title="Ventas mensuales en la selección",\n                    labels={"mes": "Mes", "ventas": "Ventas (COP)"})\nfig_serie.update_layout(hovermode="x unified")\nst.plotly_chart(fig_serie, use_container_width=True, key="grafico_evolucion")\n\nst.subheader("3 · ¿Qué categorías concentran las ventas?")\nranking = (filtrado.groupby("categoria", as_index=False)["ventas"]\n           .sum().sort_values("ventas", ascending=False))\nfig_cat = px.bar(ranking, x="categoria", y="ventas", color="categoria",\n                 title="Ventas por categoría",\n                 labels={"categoria": "Categoría", "ventas": "Ventas (COP)"})\nfig_cat.update_layout(showlegend=False)\nst.plotly_chart(fig_cat, use_container_width=True, key="grafico_categorias")\n'
print("Bloque preparado:", "bloque_visualizaciones")

Bloque preparado: bloque_visualizaciones


## 7. Narrativa, transparencia y descarga
**Paso 6.** Añadimos una lectura automatizada **descriptiva**, sin afirmar causalidad, y una opción de descargar los registros filtrados.

**Criterio de comunicación:** una cifra o un gráfico debe ayudar a responder una pregunta de la audiencia, no solo decorar la aplicación.

In [19]:
# BLOQUE DIDÁCTICO: bloque_cierre
# Se guarda como texto Python; todavía NO se modifica app_streamlit.py.
bloque_cierre = '# La lectura descriptiva se basa en el ranking de la MISMA selección filtrada.\nst.subheader("4 · Lectura de resultados")\ncategoria_lider = ranking.iloc[0]["categoria"]\nparticipacion = 100 * ranking.iloc[0]["ventas"] / ranking["ventas"].sum()\nst.success(\n    f"En la selección actual, **{categoria_lider}** concentra "\n    f"el **{participacion:.1f}%** del valor vendido. "\n    "Es una descripción de los registros filtrados, no una explicación causal."\n)\nst.caption("Fuente: datos sintéticos generados para fines docentes. "\n           "Los resultados cambian al modificar los filtros.")\nst.download_button("Descargar registros filtrados (CSV)",\n                   data=filtrado.to_csv(index=False).encode("utf-8-sig"),\n                   file_name="ventas_filtradas.csv", mime="text/csv")\n'
print("Bloque preparado:", "bloque_cierre")

Bloque preparado: bloque_cierre


### 7.1. Construir el archivo completo (sin duplicaciones)
**Paso 7.** Ejecuta esta celda después de preparar los cuatro bloques. Se utiliza `write_text`, que **reemplaza** el archivo anterior en lugar de añadir contenido. Puedes ejecutarla varias veces: el código no se duplicará.

**Control de calidad:** comprobamos que «Panorama general» aparezca una sola vez y que existan exactamente dos gráficos Plotly.

In [20]:
from pathlib import Path

# Unimos las cuatro partes del programa en el orden pedagógico previsto.
# El modo de escritura de write_text SOBRESCRIBE el archivo completo.
# A diferencia de %%writefile -a, nunca añade una copia al final.
codigo_completo = "\n\n".join([
    bloque_inicio,
    bloque_filtros,
    bloque_visualizaciones,
    bloque_cierre,
])

# Verificaciones sencillas para detectar duplicaciones antes de abrir Streamlit.
assert codigo_completo.count('st.subheader("1 · Panorama general")') == 1, "Indicadores duplicados"
assert codigo_completo.count('st.plotly_chart(') == 2, "Número incorrecto de gráficos"
assert codigo_completo.count('st.subheader("4 · Lectura de resultados")') == 1, "Lectura duplicada"

Path("app_streamlit.py").write_text(codigo_completo, encoding="utf-8")
print("Aplicación generada correctamente: app_streamlit.py")
print("Panorama general: 1 bloque | Gráficos Plotly: 2 | Lectura: 1 bloque")

Aplicación generada correctamente: app_streamlit.py
Panorama general: 1 bloque | Gráficos Plotly: 2 | Lectura: 1 bloque


## 8. Ejecutar Streamlit desde Google Colab
**Paso 8.** Inicia el servidor en segundo plano. Esta celda detiene **únicamente** el proceso guardado en la variable `proceso_streamlit` si ya lo habías iniciado en este mismo entorno.

**Paso 8.** Ejecuta la siguiente celda y haz clic en el enlace generado para abrir la aplicación. **No cierres el entorno de ejecución de Colab:** si se desconecta, la aplicación dejará de estar disponible.

> El proxy de Colab puede cambiar su comportamiento. Si el enlace no abre o aparece un error de conexión, comparte la captura para revisar el entorno antes de continuar. Este notebook **no despliega una aplicación pública permanente**.

In [21]:
import subprocess
import time
import os

if "proceso_streamlit" in globals() and proceso_streamlit.poll() is None:
    proceso_streamlit.terminate()
    try:
        proceso_streamlit.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proceso_streamlit.kill()

registro_streamlit = open("streamlit.log", "w")
proceso_streamlit = subprocess.Popen(
    ["streamlit", "run", "app_streamlit.py", "--server.port", "8501",
     "--server.address", "0.0.0.0", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=registro_streamlit, stderr=subprocess.STDOUT
)
time.sleep(5)
print("Servidor activo:", proceso_streamlit.poll() is None)
print("Archivo de registro: streamlit.log")

Servidor activo: True
Archivo de registro: streamlit.log


In [22]:
from google.colab import output
url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print("Abre la aplicación en una pestaña nueva:", url)

Abre la aplicación en una pestaña nueva: https://8501-m-s-usc1b5-6jqetrubn47u-b.us-central1.prod.r.colab.dev


### 8.1. Si la aplicación muestra un error
1. Comprueba que los cuatro bloques y la celda 7.1 se ejecutaron **en orden**. Se pueden volver a ejecutar sin duplicar el archivo.
2. Consulta el registro del servidor.
3. Si corriges `app_streamlit.py`, Streamlit normalmente detectará los cambios; recarga la pestaña si es necesario.
4. Si la sesión de Colab se reinicia, vuelve a ejecutar el notebook desde el inicio.

In [11]:
from pathlib import Path
if Path("streamlit.log").exists():
    print(Path("streamlit.log").read_text(encoding="utf-8", errors="replace")[-4000:])
else:
    print("Todavía no se ha creado el registro. Ejecuta primero la sección 8.")

## 9. Prueba guiada y discusión
Abre la aplicación y verifica, en este orden:

| Prueba | Acción | Resultado esperado |
|---|---|---|
| A | Cambiar de «Todas» a «Cali» | Cambian indicadores y gráficos |
| B | Seleccionar solo «Hogar» | El gráfico de categorías muestra únicamente Hogar |
| C | Mover el slider hasta el mes 6 | Se excluyen registros posteriores a junio |
| D | Escribir `tec` en la búsqueda | Se filtran categorías que contienen ese texto |
| E | Pulsar «Restablecer filtros» | Regresan los valores iniciales |
| F | Descargar CSV | El archivo contiene solo los registros visibles |

**Preguntas para discusión**
- ¿Qué decisiones de interfaz facilitan la comprensión a una audiencia no técnica?
- ¿Cuándo sería mejor una tabla que un gráfico?
- ¿Por qué debemos declarar que estos datos son sintéticos?
- ¿Qué controles podrían generar interpretaciones confusas?

**Validación adicional:** después de escribir «Hogar» con ciudad Bogotá y mes 6, comprueba que los indicadores del único «Panorama general» coincidan con los gráficos y el CSV filtrado. No debe aparecer un segundo bloque de indicadores al final.


## 10. Actividad práctica individual (15–20 minutos)
**Reto:** modifica la aplicación **sin copiar un gráfico adicional del docente**.

1. Añade un control para elegir la métrica del gráfico de categorías: **ventas** o **unidades**.
2. Ajusta el título y el eje vertical según la métrica seleccionada.
3. Escribe dos frases interpretativas: un hallazgo verificable y una limitación del análisis.
4. Demuestra que el nuevo control responde correctamente a los filtros existentes.

**Entrega para revisión en clase:** muestra la aplicación funcionando y explica qué líneas cambiaste.

**Pista mínima:** `st.selectbox("Métrica", ["ventas", "unidades"])` y `groupby(...)[metrica].sum()`.

**Criterios de revisión:** funcionamiento, coherencia de etiquetas, interpretación, claridad y reproducibilidad.

## 11. Descargar y conservar el trabajo
**Paso 10.** Descarga los dos archivos. Para reproducir la aplicación en otro entorno, deben permanecer **en la misma carpeta**.

**Nota:** el archivo `.py` contiene el código; el `.csv` contiene los datos. El notebook es la guía para reconstruir el flujo.

In [12]:
from google.colab import files
files.download("app_streamlit.py")
files.download("ventas_streamlit.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Cierre de la sesión
**Idea central:** Plotly aporta visualizaciones interactivas; Streamlit permite **organizar esas visualizaciones dentro de una aplicación**, junto con controles, indicadores, explicaciones y descargas.

**Reproducibilidad:** conserva el `.ipynb`, el `.py`, el `.csv` y las decisiones tomadas durante el desarrollo.

**Antes de aprobar el material:** ejecutar todo en una sesión limpia de Google Colab, comprobar el enlace de la aplicación y revisar el comportamiento de los filtros. Los servicios y proxies de Colab pueden variar con el tiempo.